In [1]:
# Dataset 必须满足__len__, __getitem__两个魔法方法，反过来，满足这两个魔法方法的就是Dataset

In [2]:
import torch
import numpy as np
from torchvision import transforms

In [3]:
# 获取图片路径
import glob

In [12]:
from PIL import Image

In [23]:
all_img_path = glob.glob(r'C:\Users\Administrator\Desktop\PythonCode\PyTorch\dataset\*.jpg')

In [19]:
# 建立类别和索引的之间的映射关系
species = ['cloudy', 'rain', 'shine', 'sunrise']
species_to_idx = dict((c, i) for i, c in enumerate(species))

In [20]:
species_to_idx.items

<function dict.items>

In [21]:
# 交换key和value的顺序
species_to_idx = dict((i, c) for i, c in species_to_idx.items())

In [22]:
species_to_idx

{'cloudy': 0, 'rain': 1, 'shine': 2, 'sunrise': 3}

In [25]:
# 生成所有图片的labels
all_labels = []

for img in all_img_path:
    for i, c in enumerate(species):
        if c in img:
            all_labels.append(i)

In [26]:
all_labels

[0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,


In [29]:
 # 借助ndarray的索引取值方法，打乱数据
index = np.random.permutation(len(all_img_path))

all_img_path = np.array(all_img_path)[index]
all_labels = np.array(all_labels)[index]

In [30]:
all_img_path[:10]

array(['C:\\Users\\Administrator\\Desktop\\PythonCode\\PyTorch\\dataset\\sunrise85.jpg',
       'C:\\Users\\Administrator\\Desktop\\PythonCode\\PyTorch\\dataset\\rain106.jpg',
       'C:\\Users\\Administrator\\Desktop\\PythonCode\\PyTorch\\dataset\\rain87.jpg',
       'C:\\Users\\Administrator\\Desktop\\PythonCode\\PyTorch\\dataset\\shine29.jpg',
       'C:\\Users\\Administrator\\Desktop\\PythonCode\\PyTorch\\dataset\\rain134.jpg',
       'C:\\Users\\Administrator\\Desktop\\PythonCode\\PyTorch\\dataset\\cloudy96.jpg',
       'C:\\Users\\Administrator\\Desktop\\PythonCode\\PyTorch\\dataset\\cloudy207.jpg',
       'C:\\Users\\Administrator\\Desktop\\PythonCode\\PyTorch\\dataset\\sunrise338.jpg',
       'C:\\Users\\Administrator\\Desktop\\PythonCode\\PyTorch\\dataset\\cloudy257.jpg',
       'C:\\Users\\Administrator\\Desktop\\PythonCode\\PyTorch\\dataset\\cloudy250.jpg'],
      dtype='<U72')

In [31]:
all_labels[:10]

array([3, 1, 1, 2, 1, 0, 0, 3, 0, 0])

In [38]:
# 手动划分训练数据和测试数据
split = int(len(all_img_path) * 0.8)

In [39]:
train_imgs = all_img_path[:split]
train_labels = all_labels[:split]

test_imgs = all_img_path[split:]
test_labels = all_labels[split:]

In [51]:
transform = transforms.Compose([
    transforms.Resize((96, 96)),
    transforms.ToTensor()
])

In [132]:
# 自定义Dataset
class MyDataset(torch.utils.data.Dataset):
    def __init__(self, img_paths, labels, transform):
        self.imgs = img_paths
        self.labels = labels
        self.transforms = transform

    def __getitem__(self, index):
        # 根据index获取item
        img_path = self.imgs[index]
        label = self.labels[index]

        # 通过PIL的Image读取图片
        img = Image.open(img_path)
        data = self.transforms(img)

        if np.array(img).shape[-1] == 3:
            # 注意：模型训练要求y必须是long类型
            # torch.from_numpy接收的参数必须是个nparray
            return data, torch.from_numpy(np.array(label)).long()
        else:
            print(img_path)

        

    def __len__(self):
        return len(self.imgs)

    @staticmethod
    def collate_fn(batch):
        #batch是个列表，长度是batch_size，
        #列表的每个元素是一个元组(x，y)
        # [(x1,y1),(x2, y2),(x3, y3)...]
        # collate_fn的作用，就是把所有的x放到一起，所有的y放到一起.
        #把batch中None过滤掉即可.
        batch = [sample for sample in batch if sample is not None]
        #简单方法，直接默认的collate方法
        # from torch.utils.data.dataloader import default_collate
        # return default_collate(batch)
        # 自定义方法
        imgs, labels = zip(*batch)
        return torch.stack(imgs, 0), torch.stack(labels, 0)

In [117]:
dataset = MyDataset(all_img_path, all_labels, transform)

In [125]:
train_ds = MyDataset(train_imgs, train_labels, transform)
test_ds = MyDataset(test_imgs, test_labels, transform)


train_dl = torch.utils.data.DataLoader(train_ds, batch_size=16, shuffle=True, collate_fn=MyDataset.collate_fn, drop_last=True)
test_dl = torch.utils.data.DataLoader(test_ds, batch_size=16 * 2, collate_fn=MyDataset.collate_fn, drop_last=True)

In [126]:
import torch.nn as nn
import torch.nn.functional as F

In [127]:
# 添加BN层.
# 定义模型
class Net(torch.nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(3, 16, 3)   # 16 * 94 * 94
        self.bn1 = nn.BatchNorm2d(16)
        self.pool = nn.MaxPool2d(2, 2)     # 16 * 47 * 47
        
        self.conv2 = nn.Conv2d(16, 32, 3)  # 32 * 45 * 45  -> pooling -> 32 * 22 * 22
        self.bn2 = nn.BatchNorm2d(32)
        self.conv3 = nn.Conv2d(32, 64, 3)  # 64 * 20 * 20  -> pooling -> 64 * 10 * 10
        self.bn3 = nn.BatchNorm2d(64)
        self.dropout = nn.Dropout()
        
        # batch , channel, height, width, 64, 
        self.fc1 = nn.Linear(64 * 10 * 10, 1024)
        self.bn_fc1 = nn.BatchNorm1d(1024)
        self.fc2 = nn.Linear(1024, 256)
        self.bn_fc2 = nn.BatchNorm1d(256)
        self.fc3 = nn.Linear(256, 4)
        
    def forward(self, x):
        x = self.pool(F.relu(self.conv1(x)))
        x = self.bn1(x)
        x = self.pool(F.relu(self.conv2(x)))
        x = self.bn2(x)
        x = self.pool(F.relu(self.conv3(x)))
        x = self.bn3(x)
        # x.view(-1, 64 * 10 * 10)
        x = nn.Flatten()(x)
        x = F.relu(self.fc1(x))
        x = self.bn_fc1(x)
        x = self.dropout(x)
        x = F.relu(self.fc2(x))
        x = self.bn_fc2(x)
        x = self.dropout(x)
        x = self.fc3(x)
        return x

In [128]:
device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')

In [129]:
model = Net()
# 把model拷到gpu上
if torch.cuda.is_available():
    model.to(device)
    
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
loss_fn = nn.CrossEntropyLoss()


In [130]:
def fit(epoch, model, train_loader, test_loader):
    correct = 0
    total = 0
    running_loss = 0
    
    # 因为bn和dropout需要手动指定训练模式和推理模式
    model.train()
    for x, y in train_loader:
        # 把数据放到GPU上去. 
        x, y = x.to(device), y.to(device)
        y_pred = model(x)
        loss = loss_fn(y_pred, y)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        with torch.no_grad():
            y_pred = torch.argmax(y_pred, dim=1)
            correct += (y_pred == y).sum().item()
            total += y.size(0)
            running_loss += loss.item()
            
    epoch_loss = running_loss / len(train_loader.dataset)
    epoch_acc = correct / total
        
    # 测试过程
    test_correct = 0
    test_total = 0
    test_running_loss = 0
    model.eval()
    with torch.no_grad():
        for x, y in test_loader:
            x, y = x.to(device), y.to(device)
            y_pred = model(x)
            loss = loss_fn(y_pred, y)
            y_pred = torch.argmax(y_pred, dim=1)
            test_correct += (y_pred == y).sum().item()
            test_total += y.size(0)
            test_running_loss += loss.item()
    test_epoch_loss = test_running_loss / len(test_loader.dataset)
    test_epoch_acc = test_correct / test_total

    print('epoch: ', epoch,
         'loss: ', round(epoch_loss, 3),
         'accuracy: ', round(epoch_acc, 3),
         'test_loss: ', round(test_epoch_loss, 3),
         'test_accuracy: ', round(test_epoch_acc, 3))
    return epoch_loss, epoch_acc, test_epoch_loss, test_epoch_acc

In [131]:
epochs = 10
train_loss = []
train_acc = []
test_loss = []
test_acc = []
for epoch in range(epochs):
    epoch_loss, epoch_acc, test_epoch_loss, test_epoch_acc = fit(epoch, model, train_dl, test_dl)
    train_loss.append(epoch_loss)
    train_acc.append(epoch_acc)
    
    test_loss.append(epoch_loss)
    test_acc.append(epoch_acc)

C:\Users\Administrator\Desktop\PythonCode\PyTorch\dataset\cloudy71.jpg
C:\Users\Administrator\Desktop\PythonCode\PyTorch\dataset\shine131.jpg
C:\Users\Administrator\Desktop\PythonCode\PyTorch\dataset\shine127.jpg
C:\Users\Administrator\Desktop\PythonCode\PyTorch\dataset\cloudy66.jpg
C:\Users\Administrator\Desktop\PythonCode\PyTorch\dataset\rain141.jpg
epoch:  0 loss:  0.045 accuracy:  0.722 test_loss:  0.015 test_accuracy:  0.825
C:\Users\Administrator\Desktop\PythonCode\PyTorch\dataset\cloudy66.jpg
C:\Users\Administrator\Desktop\PythonCode\PyTorch\dataset\shine127.jpg
C:\Users\Administrator\Desktop\PythonCode\PyTorch\dataset\shine131.jpg
C:\Users\Administrator\Desktop\PythonCode\PyTorch\dataset\cloudy71.jpg
C:\Users\Administrator\Desktop\PythonCode\PyTorch\dataset\rain141.jpg
epoch:  1 loss:  0.03 accuracy:  0.821 test_loss:  0.011 test_accuracy:  0.834
C:\Users\Administrator\Desktop\PythonCode\PyTorch\dataset\cloudy66.jpg
C:\Users\Administrator\Desktop\PythonCode\PyTorch\dataset\shin